In [1]:
import os
import glob
import torch
import torch.nn as nn
import onnx

from finn.util.basic import make_build_dir
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper

# Dataflow transforms from FINN
from finn.transformation.fpgadataflow.convert_to_hls import ConvertToHLSLayer
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
from finn.transformation.fpgadataflow.export import ExportRTL

# Basic FINN transformations
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.streamline import Streamline
import finn.transformation.streamline.absorb as absorb
from qonnx.transformation.general import (
    RemoveUnusedTensors,
    GiveUniqueNodeNames,
    GiveReadableTensorNames,
    RemoveStaticGraphInputs,
)
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount

# Import your full UNet model
from split_unet import UNetSplit

###############################################################################
# 1) Submodule that outputs ONLY the final encoder feature
###############################################################################
class EncoderOnly(nn.Module):
    def __init__(self, unet_split):
        super().__init__()
        # Reuse the existing encoder from UNetSplit
        self.encoder = unet_split.encoder

    def forward(self, x):
        # The full encoder returns (encoded_x, skip_list)
        encoded_x, skip_list = self.encoder(x)
        # For hardware acceleration, we return only encoded_x
        return encoded_x

###############################################################################
# 2) Helper to remove any leftover "Mul" node at the end (optional)
###############################################################################
def remove_final_mul_op(model: ModelWrapper):
    """
    Sometimes the last node is a leftover Mul that can't be dataflow-partitioned.
    We rewire the final output to the Mul's input, removing the Mul node.
    """
    graph = model.graph
    fout_name = graph.output[0].name
    last_node = None
    for node in graph.node:
        if fout_name in node.output:
            last_node = node
            break
    if last_node is not None and last_node.op_type == "Mul":
        mul_input = last_node.input[0]
        print("Removing final Mul node, rewiring output to:", mul_input)
        graph.output[0].name = mul_input
        keep_nodes = [n for n in graph.node if n != last_node]
        graph.ClearField("node")
        graph.node.extend(keep_nodes)
    else:
        print("No final Mul node found.")
    return model

###############################################################################
# 3) Rename final output to avoid any leftover name collisions
###############################################################################
def rename_final_output(model: ModelWrapper, new_name="final_out"):
    old_name = model.graph.output[0].name
    print(f"Renaming final output from {old_name} to {new_name}")
    model.graph.output[0].name = new_name

    # Remove references to old_name in graph.input
    keep_inp = []
    for inp in model.graph.input:
        if inp.name != old_name:
            keep_inp.append(inp)
    del model.graph.input[:]
    model.graph.input.extend(keep_inp)

    # Remove references to old_name in value_info
    new_vi = []
    for vi in model.graph.value_info:
        if vi.name != old_name:
            new_vi.append(vi)
    del model.graph.value_info[:]
    model.graph.value_info.extend(new_vi)

    return model

###############################################################################
# 4) Main build script
###############################################################################
def main():
    # Create unique build folder inside /tmp/finn_dev_*
    build_dir = make_build_dir("UNetEncoder_build")

    MODEL_WEIGHTS = "./best_unet_weights.pth"  # Path to your trained UNet weights

    # 1) Load full UNet, weights
    unet_full = UNetSplit(in_ch=1, out_ch=1)
    unet_full.load_state_dict(torch.load(MODEL_WEIGHTS, map_location="cpu"))
    unet_full.eval()

    # 2) Wrap only the encoder
    encoder_only = EncoderOnly(unet_full).eval()

    # 3) Export that submodule to ONNX
    export_onnx_path = os.path.join(build_dir, "unet_encoder_export.onnx")
    dummy_input = torch.randn(1, 1, 128, 128)  # fixed-size input
    print(f"Exporting encoder-only to ONNX with shape {dummy_input.shape} ...")
    export_qonnx(encoder_only, dummy_input, export_onnx_path)
    qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)

    # 4) Load into FINN + transformations
    model = ModelWrapper(export_onnx_path)
    model = model.transform(ConvertQONNXtoFINN())
    model = model.transform(InferShapes())
    model = model.transform(FoldConstants())
    model = model.transform(Streamline())
    model = model.transform(LowerConvsToMatMul())
    model = model.transform(ConvertBipolarMatMulToXnorPopcount())
    model = model.transform(Streamline())
    model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
    model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
    model = model.transform(InferDataLayouts())
    model = model.transform(RemoveUnusedTensors())
    model = model.transform(RemoveStaticGraphInputs())
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())

    # 5) Remove final Mul (optional), rename final output
    model = remove_final_mul_op(model)
    model = rename_final_output(model, "encoder_out")
    streamlined_onnx = os.path.join(build_dir, "unet_encoder_streamlined.onnx")
    model.save(streamlined_onnx)
    print(f"Streamlined single-output encoder model saved to {streamlined_onnx}")

    # 6) Convert entire graph to dataflow HLS layers (no partial partition)
    model = model.transform(ConvertToHLSLayer())
    model = model.transform(SetExecMode("rtl"))
    hls_prep_onnx = os.path.join(build_dir, "encoder_df_before_sim.onnx")
    model.save(hls_prep_onnx)

    # 7) Prepare C++Sim
    model = model.transform(PrepareCppSim())
    cppsim_onnx = os.path.join(build_dir, "encoder_df_cppsim.onnx")
    model.save(cppsim_onnx)

    # 8) HLS synth
    model = model.transform(HLSSynthIP())
    hls_synth_onnx = os.path.join(build_dir, "encoder_df_hls.onnx")
    model.save(hls_synth_onnx)

    # 9) Create Stitched IP for Pynq-Z2
    model = model.transform(CreateStitchedIP(
        platform="Pynq-Z2", 
        period_ns=10.0,  # 100 MHz
        generate_bitfile=True
    ))
    stitched_onnx = os.path.join(build_dir, "encoder_df_stitched.onnx")
    model.save(stitched_onnx)

    # 10) Export RTL + Make Python driver
    model = model.transform(ExportRTL())
    model = model.transform(MakePYNQDriver(platform="Pynq-Z2"))
    final_onnx = os.path.join(build_dir, "encoder_df_final.onnx")
    model.save(final_onnx)

    print("Dataflow build complete.")
    print(f"Final dataflow ONNX: {final_onnx}")
    print("Check the build folder for bitfile + Python driver for your Pynq-Z2.")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'finn.transformation.fpgadataflow.convert_to_hls'